In [1]:

# ! pip install -r ../../../../requirements.txt -U

In [2]:
from agent_framework.azure import AzureAIAgentClient, AzureAIAgentsProvider
from azure.identity import AzureCliCredential

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    BingGroundingTool,
    BingGroundingSearchToolParameters,
    BingGroundingSearchConfiguration,
)
from azure.core.exceptions import ResourceNotFoundError



In [3]:

import os
import base64
from dotenv import load_dotenv

In [4]:
AgentName = "Search-Agent"
AgentInstructions = """You are a helpful assistant that can search the web for current information.
            Use the Bing search tool to find up-to-date information and provide accurate, well-sourced answers.
            Always cite your sources when possible. Add something fun at the end of your answer."""

In [5]:
# 1. Resolve the Foundry project connection ID from connection name candidates
env_connection_name = os.environ["BING_CONNECTION_NAME"]

with AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=AzureCliCredential(),
) as project:
    bing_connection = None
    try:
        bing_connection = project.connections.get(env_connection_name)
        print(f"Resolved connection: {bing_connection.name} -> {bing_connection.id}")
    except ResourceNotFoundError:
        pass

    if bing_connection is None:
        available = [c.name for c in project.connections.list()]
        raise RuntimeError(
            f"No matching Bing connection found. Tried: {env_connection_name}. "
            f"Available project connections: {available}"
        )

    # 2. Check if agent exists and get latest version
    agent_name = AgentName
    latest_version = None
    
    try:
        # Try to get the latest version of the agent
        versions = list(project.agents.list_versions(agent_name=agent_name))
        if versions:
            latest_agent = versions[0]  # Get the latest version
            latest_version = int(latest_agent.version) if hasattr(latest_agent, 'version') else latest_agent.version
            next_version = latest_version + 1
            print(f"Agent '{agent_name}' exists. Latest version: {latest_version}. Creating new version: {next_version}")
    except ResourceNotFoundError:
        print(f"Agent '{agent_name}' does not exist. Creating new agent with version: {next_version}")

    # 3. Create a new agent version with Bing grounding tool attached (sync path)
    model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME")
    agent = project.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=model_name,
            instructions=AgentInstructions,
            tools=[
                BingGroundingTool(
                    bing_grounding=BingGroundingSearchToolParameters(
                        search_configurations=[
                            BingGroundingSearchConfiguration(
                                project_connection_id=bing_connection.id
                            )
                        ]
                    )
                )
            ],
        ),
        description="Bing grounded search agent",
    )

    print(f"\n✓ Agent version {agent.version} created successfully")
    print(f"  - Agent ID: {agent.id}")
    print(f"  - Agent Name: {agent.name}")
    print(f"  - Version: {agent.version}")
    if latest_version:
        print(f"  - Previous version: {latest_version}")

Resolved connection: hanabingsearchj9kdyn -> /subscriptions/d6233897-5c9f-47f9-8507-6d4ada2d5183/resourceGroups/AgenticAI-AzureMasterclass/providers/Microsoft.CognitiveServices/accounts/hana-foundry-test/projects/proj-default/connections/hanabingsearchj9kdyn
Agent 'Search-Agent' exists. Latest version: 1. Creating new version: 2

✓ Agent version 2 created successfully
  - Agent ID: Search-Agent:2
  - Agent Name: Search-Agent
  - Version: 2
  - Previous version: 1


In [6]:
UserQuestion = "when was the city of Casablanca founded"

In [7]:
# from azure.identity import DefaultAzureCredential

# my_endpoint = "https://hana-foundry-test.services.ai.azure.com/api/projects/proj-default"

project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=AzureCliCredential(),
)

my_agent = agent_name
my_version = "5"

openai_client = project_client.get_openai_client()

# Reference the agent to get a response
response = openai_client.responses.create(
    input=[{"role": "user", "content": UserQuestion}],
    extra_body={"agent_reference": {"name": agent_name, "version": agent.version, "type": "agent_reference"}},
)

print(f"Response output: {response.output_text}")

Response output: The site of modern Casablanca has been inhabited since ancient times, so there isn’t a single, precise “founding year” like many younger cities.

Key historical points:

- A Berber settlement called **Anfa** existed on the site by about the **10th century BC**.【5:0†source】  
- It later became a Phoenician and then Roman trading port; the Romans occupied the area around **15 BC**.【5:0†source】  
- The Portuguese destroyed Anfa in **1468**, then built a new fortified town called **Casa Branca** in **1515**.【5:3†source】  
- After the 1755 earthquake, Sultan Sidi Mohammed ben Abdallah rebuilt the town in the **late 18th century** and named it **ad-Dār al-Bayḍāʾ** (Arabic for “Casablanca”).【5:3†source】

So:  
- As a settlement (Anfa): roughly **10th century BC**.  
- As a European-style port town (Casa Branca/Casablanca): effectively **16th–18th century**, with major rebuilding in the **late 1700s**.

Fun extra: If you ever stroll through modern Casablanca, you’re walking on